# Telecom Customer Intelligence Platform

## Notebook 04 â€“ Business Insights Analysis

### Objective

Analyze customer behavior using the Silver dataset and generate actionable business insights.

### Input

Silver Layer

### Output

Business KPIs, Customer Insights and Executive Recommendations

In [ ]:
# Local Jupyter / GitHub compatibility (no-op on Databricks)
import os
import sys
from pathlib import Path

if "DATABRICKS_RUNTIME_VERSION" not in os.environ:
    _root = next(
        (
            p
            for p in [Path.cwd(), *Path.cwd().parents]
            if (p / "src" / "utils" / "databricks_compat.py").exists()
        ),
        None,
    )
    if _root and str(_root / "src") not in sys.path:
        sys.path.insert(0, str(_root / "src"))
    from utils.databricks_compat import inject_globals

    inject_globals(globals())


In [ ]:

# Initialization

import sys

PROJECT_ROOT = "/Workspace/Users/thakurshivek777@gmail.com/Customer-Churn-Platform"
SRC_PATH = f"{PROJECT_ROOT}/src"

if SRC_PATH not in sys.path:
    sys.path.append(SRC_PATH)

print(SRC_PATH)
print(sys.path)

In [ ]:

from pyspark.sql import functions as F

from config.settings import SILVER_TABLE
from features.churn_analysis import analyze_churn

In [ ]:

silver_df = spark.read.table(SILVER_TABLE)

display(silver_df.limit(5))

In [ ]:

total_customers = silver_df.count()

churned_customers = silver_df.filter(
    F.col("Churn") == "Yes"
).count()

active_customers = silver_df.filter(
    F.col("Churn") == "No"
).count()

churn_rate = (
    churned_customers / total_customers
) * 100

avg_monthly = silver_df.select(
    F.avg("MonthlyCharges")
).first()[0]

avg_tenure = silver_df.select(
    F.avg("tenure")
).first()[0]

In [ ]:

print("=" * 70)
print("EXECUTIVE KPI DASHBOARD")
print("=" * 70)

print(f"Total Customers          : {total_customers}")
print(f"Active Customers         : {active_customers}")
print(f"Churned Customers        : {churned_customers}")
print(f"Churn Rate (%)           : {churn_rate:.2f}")
print(f"Average Monthly Charges  : ${avg_monthly:.2f}")
print(f"Average Tenure (Months)  : {avg_tenure:.2f}")

In [ ]:

# Analyze Gender Distribution
gender_summary = (
    silver_df
    .groupBy("gender", "Churn")
    .count()
    .orderBy("gender", "Churn")
)

display(gender_summary)

In [ ]:

# Calculate Churn Rate by Gender
gender_churn = (
    silver_df
    .groupBy("gender")
    .agg(
        F.count("*").alias("Total_Customers"),
        F.sum(
            F.when(F.col("Churn") == "Yes", 1).otherwise(0)
        ).alias("Churned_Customers")
    )
)

gender_churn = (
    gender_churn
    .withColumn(
        "Churn_Rate",
        F.round(
            (F.col("Churned_Customers") / F.col("Total_Customers")) * 100,
            2
        )
    )
)

display(gender_churn)


## Business Insight

**Question**

Does gender influence customer churn?

**Observation**

(We'll write this after seeing the output.)

**Recommendation**

(We'll decide based on evidence.)


## Business Question

Do senior citizens churn more than non-senior customers?

In [ ]:

senior_churn = (
    silver_df
    .groupBy("SeniorCitizen")
    .agg(
        F.count("*").alias("Total_Customers"),
        F.sum(
            F.when(F.col("Churn") == "Yes", 1).otherwise(0)
        ).alias("Churned_Customers")
    )
)

In [ ]:

senior_churn = (
    senior_churn
    .withColumn(
        "Churn_Rate",
        F.round(
            (F.col("Churned_Customers") / F.col("Total_Customers")) * 100,
            2
        )
    )
)

display(senior_churn)

In [ ]:

display(senior_churn)


## Business Insight

### Business Question

Do senior citizens churn more than non-senior customers?

### Evidence

- Non-Senior Churn Rate : **23.61%**
- Senior Citizen Churn Rate : **41.68%**

Difference : **18.07 percentage points**

### Business Insight

Senior citizens are significantly more likely to churn than non-senior customers. This indicates that age group is a strong factor associated with customer churn.

### Business Impact

ðŸ”´ High

### Recommendation

Develop targeted retention strategies for senior customers, such as simplified support channels, loyalty benefits, personalized service plans, and proactive customer outreach.

In [ ]:

contract_churn = analyze_churn(silver_df, "Contract")
display(contract_churn)


## Business Insight

### Business Question

Does contract type influence customer churn?

### Evidence

| Contract | Churn Rate |
|----------|-----------:|
| Month-to-month | 42.71% |
| One year | 11.27% |
| Two year | 2.83% |

### Business Insight

Contract type is one of the strongest indicators of customer churn. Customers on month-to-month contracts churn at a significantly higher rate than customers on long-term contracts.

### Business Impact

ðŸ”´ Very High

### Recommendation

Increase customer retention by encouraging month-to-month customers to migrate to annual or two-year contracts through discounts, loyalty rewards, bundled services, or exclusive offers.


# Internet Service Analysis

## Business Question

Does the type of internet service influence customer churn?

In [ ]:

internet_churn = analyze_churn(
    silver_df,
    "InternetService"
)

display(internet_churn)


## Business Insight

### Business Question

Does Internet Service influence customer churn?

### Evidence

| Internet Service | Churn Rate |
|-----------------|-----------:|
| Fiber optic | 41.89% |
| DSL | 18.96% |
| No Internet | 7.40% |

### Business Insight

Customers using Fiber Optic internet experience the highest churn rate, more than double that of DSL users.

### Business Impact

ðŸ”¥ Critical

### Recommendation

Investigate Fiber Optic service quality, pricing, customer support, and network stability. Prioritize retention campaigns for Fiber Optic customers.


# Online Security Analysis

## Business Question

Does having Online Security reduce customer churn?

In [ ]:

security_churn = analyze_churn(
    silver_df,
    "OnlineSecurity"
)

display(security_churn)


## Business Insight

### Business Question

Does Online Security reduce churn?

### Evidence

Without Online Security : 41.77%

With Online Security : 14.61%

### Business Insight

Customers without Online Security churn nearly three times more often than customers who subscribe to the service.

### Business Impact

ðŸ”¥ Critical

### Recommendation

Bundle Online Security into premium internet plans or provide discounted introductory offers.


# Tech Support Analysis

## Business Question

Does Tech Support influence customer retention?

In [ ]:

tech_support_churn = analyze_churn(
    silver_df,
    "TechSupport"
)

display(tech_support_churn)


## Business Insight

### Business Question

Does Tech Support improve customer retention?

### Evidence

Without Tech Support : 41.64%

With Tech Support : 15.17%

### Business Insight

Customers who subscribe to Tech Support are significantly more likely to remain with the company.

### Business Impact

ðŸ”¥ Critical

### Recommendation

Promote Tech Support bundles and improve awareness among new customers.


# Device Protection Analysis

## Business Question

Does Device Protection reduce churn?

In [ ]:

device_churn = analyze_churn(
    silver_df,
    "DeviceProtection"
)

display(device_churn)


## Business Insight

### Business Question

Does Device Protection reduce churn?

### Evidence

Without Device Protection : 39.13%

With Device Protection : 22.50%

### Business Insight

Customers without Device Protection show considerably higher churn than customers who subscribe to the service.

### Business Impact

ðŸ”´ High

### Recommendation

Offer Device Protection during onboarding and contract renewal.


# Executive Business Findings


| Feature | Highest Risk Category | Churn Rate | Priority | Recommendation |
|---------|-----------------------|-----------:|----------|----------------|
| Gender | Female | 26.92% | ðŸŸ¢ Low | No Action |
| Senior Citizen | Yes | 41.68% | ðŸ”´ High | Target Retention Campaign |
| Contract | Month-to-month | 42.71% | ðŸ”¥ Critical | Convert to Long-Term Contracts |
| Internet Service | Fiber Optic | 41.89% | ðŸ”¥ Critical | Improve Fiber Service Quality |
| Online Security | No | 41.77% | ðŸ”¥ Critical | Bundle Security Services |
| Tech Support | No | 41.64% | ðŸ”¥ Critical | Promote Tech Support Plans |
| Device Protection | No | 39.13% | ðŸ”´ High | Offer Protection Bundles |


# Customer Risk Profiling

## Objective

Identify customer profiles that have the highest probability of churn by combining multiple business attributes.

In [ ]:

contract_internet = (
    silver_df
    .groupBy("Contract", "InternetService")
    .agg(
        F.count("*").alias("Customers"),
        F.sum(
            F.when(F.col("Churn") == "Yes", 1).otherwise(0)
        ).alias("Churned")
    )
    .withColumn(
        "Churn_Rate",
        F.round(
            (F.col("Churned") / F.col("Customers")) * 100,
            2
        )
    )
    .orderBy(F.desc("Churn_Rate"))
)

display(contract_internet)


## Business Insight

### Business Question

Which combination of Contract and Internet Service has the highest churn?

### Key Findings

- Month-to-month + Fiber Optic customers have the highest churn rate (54.61%).
- Long-term contracts dramatically reduce churn regardless of internet service.
- Two-year contract customers show extremely low churn.

### Business Recommendation

Prioritize converting Month-to-month Fiber customers into One-year or Two-year contracts using discounts and loyalty offers.

In [ ]:

senior_contract = (
    silver_df
    .groupBy("SeniorCitizen", "Contract")
    .agg(
        F.count("*").alias("Customers"),
        F.sum(
            F.when(F.col("Churn") == "Yes", 1).otherwise(0)
        ).alias("Churned")
    )
    .withColumn(
        "Churn_Rate",
        F.round(
            (F.col("Churned") / F.col("Customers")) * 100,
            2
        )
    )
    .orderBy(F.desc("Churn_Rate"))
)

display(senior_contract)


## Business Insight

### Business Question

How does Contract Type affect Senior Citizens?

### Key Findings

Senior Citizens with Month-to-month contracts have the highest churn rate (54.65%).

Long-term contracts significantly reduce churn even for Senior Citizens.

### Recommendation

Create special long-term plans and loyalty discounts specifically for Senior Citizens.

In [ ]:

support_security = (
    silver_df
    .groupBy("TechSupport", "OnlineSecurity")
    .agg(
        F.count("*").alias("Customers"),
        F.sum(
            F.when(F.col("Churn") == "Yes", 1).otherwise(0)
        ).alias("Churned")
    )
    .withColumn(
        "Churn_Rate",
        F.round(
            (F.col("Churned") / F.col("Customers")) * 100,
            2
        )
    )
    .orderBy(F.desc("Churn_Rate"))
)

display(support_security)


## Business Insight

### Business Question

How does Contract Type affect Senior Citizens?

### Key Findings

Senior Citizens with Month-to-month contracts have the highest churn rate (54.65%).

Long-term contracts significantly reduce churn even for Senior Citizens.

### Recommendation

Create special long-term plans and loyalty discounts specifically for Senior Citizens.


# Executive Customer Personas

## ðŸ”¥ Highest Risk Customer

- Senior Citizen
- Month-to-month Contract
- Fiber Optic Internet
- No Online Security
- No Tech Support
- No Device Protection

Expected churn risk:
Very High (>50%)

---

## ðŸŸ¢ Lowest Risk Customer

- Two-Year Contract
- DSL or No Internet
- Online Security = Yes
- Tech Support = Yes

Expected churn risk:
Very Low (<10%)